In [ ]:

import copy
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==================================================
# 1. Experiment configuration
# ==================================================

SEED = 42
DIM = 4
N_PATCHES = 4
BATCH_SIZE = 16
EPOCHS = 40
LR = 0.002
EMA_MOMENTUM = 0.99

OUTPUT_DIR = Path("jepa_results")
OUTPUT_DIR.mkdir(exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)

DEVICE = "cpu"

# ==================================================
# 2. Synthetic dataset
# ==================================================

def make_images(n, seed):
    generator = torch.Generator().manual_seed(seed)

    a = torch.rand(n, 1, 1, generator=generator) * 0.6 + 0.2
    b = torch.rand(n, 1, 1, generator=generator) * 0.6 + 0.2

    x = torch.linspace(0, 1, 8).view(1, 1, 8)
    y = torch.linspace(0, 1, 8).view(1, 8, 1)

    images = a * x + b * y
    return images.unsqueeze(1).float()


train_images = make_images(256, seed=42)
test_images = make_images(128, seed=123)

def patchify(images):
    return F.unfold(
        images, kernel_size=4, stride=4
    ).transpose(1, 2)

# ==================================================
# 3. Context and target encoder
# ==================================================

class TinyEncoder(nn.Module):
    def __init__(self, dim=4):
        super().__init__()

        self.patch_embed = nn.Linear(16, dim)
        self.pos_embed = nn.Parameter(
            torch.randn(1, N_PATCHES, dim) * 0.02
        )

        layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=2,
            dim_feedforward=16,
            dropout=0.0,
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            layer,
            num_layers=1,
            enable_nested_tensor=False
        )

        self.norm = nn.LayerNorm(dim)

    def forward(self, patches, indices):
        selected = patches[:, indices, :]

        tokens = (
            self.patch_embed(selected)
            + self.pos_embed[:, indices, :]
        )

        return self.norm(self.transformer(tokens))


# ==================================================
# 4. Classical predictor
# ==================================================

class ClassicalPredictor(nn.Module):
    def __init__(self, dim=4):
        super().__init__()

        self.target_pos = nn.Embedding(N_PATCHES, dim)

        self.network = nn.Sequential(
            nn.Linear(dim * 2, 16),
            nn.GELU(),
            nn.Linear(16, dim)
        )

    def forward(self, context, target_idx):

        summary = context.mean(dim=1)

        positions = torch.full(
            (context.shape[0],),
            target_idx,
            dtype=torch.long,
            device=context.device
        )

        pos = self.target_pos(positions)

        return self.network(
            torch.cat([summary, pos], dim=-1)
        )


# ==================================================
# 5. Initialize models
# ==================================================

encoder = TinyEncoder(DIM).to(DEVICE)
target_encoder = copy.deepcopy(encoder)
predictor = ClassicalPredictor(DIM).to(DEVICE)

for p in target_encoder.parameters():
    p.requires_grad_(False)

# Save initial states for later controlled
# classical-versus-quantum comparisons.
initial_state = {
    "seed": SEED,
    "encoder": copy.deepcopy(encoder.state_dict()),
    "target_encoder": copy.deepcopy(
        target_encoder.state_dict()
    ),
    "predictor": copy.deepcopy(predictor.state_dict())
}

torch.save(
    initial_state,
    OUTPUT_DIR / "initial_state.pt"
)

optimizer = torch.optim.Adam(
    list(encoder.parameters())
    + list(predictor.parameters()),
    lr=LR
)

# ==================================================
# 6. Evaluation
# ==================================================

@torch.no_grad()
def collect_predictions(images):
    encoder.eval()
    target_encoder.eval()
    predictor.eval()

    all_predictions = []
    all_targets = []

    # Evaluate every possible masked-patch position
    for start in range(0, len(images), BATCH_SIZE):
        batch = images[start:start + BATCH_SIZE]
        patches = patchify(batch)

        target_tokens = target_encoder(
            patches, list(range(N_PATCHES))
        )

        for target_idx in range(N_PATCHES):
            context_idx = [
                i for i in range(N_PATCHES)
                if i != target_idx
            ]

            context = encoder(patches, context_idx)
            prediction = predictor(context, target_idx)
            target = target_tokens[:, target_idx, :]

            all_predictions.append(prediction)
            all_targets.append(target)

    return (
        torch.cat(all_predictions, dim=0),
        torch.cat(all_targets, dim=0)
    )


def evaluate(images, constant=None):
    predictions, targets = collect_predictions(images)

    mse = F.mse_loss(predictions, targets).item()
    smooth = F.smooth_l1_loss(
        predictions, targets
    ).item()

    target_var = targets.var(
        dim=0, unbiased=False
    ).mean().item()

    prediction_var = predictions.var(
        dim=0, unbiased=False
    ).mean().item()

    result = {
        "mse": mse,
        "smooth_l1": smooth,
        "target_variance": target_var,
        "prediction_variance": prediction_var
    }

    if constant is not None:
        const_prediction = constant.expand_as(targets)

        result["constant_mse"] = F.mse_loss(
            const_prediction, targets
        ).item()

    return result


# ==================================================
# 7. Classical JEPA training
# ==================================================

print("\nStarting classical JEPA validation")
print("Training examples:", len(train_images))
print("Test examples:", len(test_images))

history = []

for epoch in range(EPOCHS):

    encoder.train()
    predictor.train()
    target_encoder.eval()

    indices = torch.randperm(len(train_images))
    total_loss = 0.0

    for start in range(0, len(indices), BATCH_SIZE):

        batch = train_images[
            indices[start:start + BATCH_SIZE]
        ]

        patches = patchify(batch)

        target_idx = torch.randint(
            0, N_PATCHES, (1,)
        ).item()

        context_idx = [
            i for i in range(N_PATCHES)
            if i != target_idx
        ]

        context = encoder(patches, context_idx)

        with torch.no_grad():
            target_tokens = target_encoder(
                patches, list(range(N_PATCHES))
            )
            target = target_tokens[:, target_idx, :]

        prediction = predictor(context, target_idx)

        loss = F.smooth_l1_loss(
            prediction, target
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Update EMA target encoder
        with torch.no_grad():
            for p, tp in zip(
                encoder.parameters(),
                target_encoder.parameters()
            ):
                tp.mul_(EMA_MOMENTUM).add_(
                    p, alpha=1 - EMA_MOMENTUM
                )

        total_loss += loss.item() * len(batch)

    train_epoch_loss = total_loss / len(train_images)

    train_eval = evaluate(train_images)
    test_eval = evaluate(test_images)

    history.append({
        "epoch": epoch,
        "train_mse": train_eval["mse"],
        "test_mse": test_eval["mse"],
        "target_variance": test_eval["target_variance"],
        "prediction_variance": test_eval[
            "prediction_variance"
        ]
    })

    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(
            f"Epoch {epoch:02d} | "
            f"Train MSE: {train_eval['mse']:.6f} | "
            f"Test MSE: {test_eval['mse']:.6f} | "
            f"Target Var: {test_eval['target_variance']:.6f}"
        )

# ==================================================
# 8. Final diagnostics
# ==================================================

train_predictions, train_targets = collect_predictions(
    train_images
)

# Mean target representation from training data only
constant_vector = train_targets.mean(
    dim=0, keepdim=True
)

train_results = evaluate(
    train_images, constant=constant_vector
)

test_results = evaluate(
    test_images, constant=constant_vector
)

print("\n" + "=" * 50)
print("FINAL CLASSICAL JEPA DIAGNOSTICS")
print("=" * 50)

for name, results in [
    ("TRAIN", train_results),
    ("TEST", test_results)
]:
    print(f"\n{name}")
    for metric, value in results.items():
        print(f"{metric:22s}: {value:.8f}")

# ==================================================
# 9. Save trained checkpoint and results
# ==================================================

torch.save({
    "seed": SEED,
    "config": {
        "dim": DIM,
        "n_patches": N_PATCHES,
        "epochs": EPOCHS,
        "learning_rate": LR,
        "ema_momentum": EMA_MOMENTUM
    },
    "encoder": encoder.state_dict(),
    "target_encoder": target_encoder.state_dict(),
    "predictor": predictor.state_dict(),
    "optimizer": optimizer.state_dict(),
    "history": history,
    "train_results": train_results,
    "test_results": test_results,
    "constant_vector": constant_vector
}, OUTPUT_DIR / "classical_jepa.pt")

print("\nSaved files:")
print("jepa_results/initial_state.pt")
print("jepa_results/classical_jepa.pt")

print("\nExperiment completed.")
